# Tema 5 – Laboratorul 5: Modele Markov ascunse (HMM)

## Exercițiul 1 – Forward

Notăm $o_{1:t}=(o_1,\dots,o_t)$ și scriem simplu $o_{1:t}$ pentru evenimentul $\{O_{1:t}=o_{1:t}\}$. Ipotezele HMM:

- (Markov) $P(X_t=j\mid X_{1:t-1},O_{1:t-1})=P(X_t=j\mid X_{t-1})=a_{X_{t-1}j}$
- (emisie) $P(O_t=o\mid X_{1:t},O_{1:t-1})=P(O_t=o\mid X_t)=b_{X_t}(o)$

### a) Inițializare

$$\alpha_1(j)=P(O_1=o_1,X_1=j)=P(X_1=j)\,P(O_1=o_1\mid X_1=j)=\pi_j\,b_j(o_1).$$

### b) Recurență

Evenimentele $\{X_{t-1}=i\}$ formează o partiție, deci, din formula probabilității totale:

$$\alpha_t(j)=P(o_{1:t},X_t=j)=\sum_i P(o_{1:t},\,X_{t-1}=i,\,X_t=j).$$

Factorizăm fiecare termen cu regula produsului:

$$P(o_{1:t},X_{t-1}=i,X_t=j)=P(o_t\mid o_{1:t-1},X_{t-1}=i,X_t=j)\;P(X_t=j\mid o_{1:t-1},X_{t-1}=i)\;P(o_{1:t-1},X_{t-1}=i).$$

1. $P(o_t\mid o_{1:t-1},X_{t-1}=i,X_t=j)=P(o_t\mid X_t=j)=b_j(o_t)$, deoarece observația curentă depinde doar de starea curentă.
2. $P(X_t=j\mid o_{1:t-1},X_{t-1}=i)=P(X_t=j\mid X_{t-1}=i)=a_{ij}$, din proprietatea Markov.
3. $P(o_{1:t-1},X_{t-1}=i)=\alpha_{t-1}(i)$, din definiție.

Rezultă

$$\alpha_t(j)=\sum_i\alpha_{t-1}(i)\,a_{ij}\,b_j(o_t)=b_j(o_t)\sum_i\alpha_{t-1}(i)\,a_{ij},\qquad t=2,\dots,T.$$

### c) Finalizare

Evenimentele $\{X_T=j\}$ formează o partiție, deci

$$P(o_{1:T})=\sum_j P(o_{1:T},X_T=j)=\sum_j\alpha_T(j).$$

---

## Exercițiul 2 – Viterbi

### a) Definirea modelului

Ordinea stărilor: D, M, U. Ordinea observațiilor: FB, B, S, NS.

In [1]:
import numpy as np

states = ["D", "M", "U"]
obs_names = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])
A = np.array([[0.0, 0.5, 0.5],
              [0.5, 0.25, 0.25],
              [0.5, 0.25, 0.25]])
B = np.array([[0.1, 0.2, 0.4, 0.3],
              [0.15, 0.25, 0.5, 0.1],
              [0.2, 0.3, 0.4, 0.1]])

notes = ["FB", "FB", "S", "B", "B", "S", "B", "B", "NS", "B", "B", "S"]
o = [obs_names.index(n) for n in notes]
T, N = len(o), len(states)

print("sume A:", A.sum(axis=1))
print("sume B:", B.sum(axis=1))
assert np.isclose(pi.sum(), 1) and np.allclose(A.sum(axis=1), 1) and np.allclose(B.sum(axis=1), 1)

sume A: [1. 1. 1.]
sume B: [1. 1. 1.]


### b) Inițializarea Viterbi

$\delta_t(j)$ este probabilitatea comună maximă a unei secvențe de stări care se termină în $j$ la momentul $t$, împreună cu observațiile până la $t$.

Inițial, având prima notă $o_1 = \text{FB}$, calculăm $\delta_1(j)=\pi_j\,b_j(o_1)$:
* $\delta_1(D) = \frac{1}{3} \cdot 0.10 = 0.0333$
* $\delta_1(M) = \frac{1}{3} \cdot 0.15 = 0.0500$
* $\delta_1(U) = \frac{1}{3} \cdot 0.20 = 0.0667$

### c) Recurența

$$\delta_t(j)=b_j(o_t)\max_i\big(\delta_{t-1}(i)\,a_{ij}\big),\qquad \psi_t(j)=\arg\max_i\big(\delta_{t-1}(i)\,a_{ij}\big).$$

Rolul lui $\psi$: $\delta_t(j)$ reține doar scorul celui mai bun traseu care se termină în $j$, nu și traseul. $\psi_t(j)$ reține starea de la $t-1$ prin care trece acel traseu optim. După ultimul pas se alege $x_T^*=\arg\max_j\delta_T(j)$, apoi se merge înapoi cu $x_{t-1}^*=\psi_t(x_t^*)$. Fără $\psi$ am cunoaște scorul maxim, dar nu și secvența.

In [2]:
delta = np.zeros((T, N))
psi = np.zeros((T, N), dtype=int)

delta[0] = pi * B[:, o[0]]
for t in range(1, T):
    cand = delta[t - 1][:, None] * A
    psi[t] = cand.argmax(axis=0)
    delta[t] = B[:, o[t]] * cand.max(axis=0)

### d) Verificare intermediară

In [3]:
print("delta_1:", dict(zip(states, delta[0].round(6).tolist())))
print("delta_2:", dict(zip(states, delta[1].round(6).tolist())))

delta_1: {'D': 0.033333, 'M': 0.05, 'U': 0.066667}
delta_2: {'D': 0.003333, 'M': 0.0025, 'U': 0.003333}


#### Verificare manuală pentru $\delta_2(D)$

Din valorile calculate pentru primul test, verificăm manual traseul optim către starea D la testul 2:
* Din $D$: $\delta_1(D) \cdot a_{DD} = 0.033333 \cdot 0.0 = 0$
* Din $M$: $\delta_1(M) \cdot a_{MD} = 0.050000 \cdot 0.5 = 0.025000$
* Din $U$: $\delta_1(U) \cdot a_{UD} = 0.066667 \cdot 0.5 = 0.033333$

Maximul dintre aceste scoruri este $0.033333$ (traseul care vine din starea U).
Înmulțim acest maxim cu probabilitatea de emisie a notei FB la un test Dificil ($b_D(\text{FB})$):
$$\delta_2(D) = 0.1 \cdot 0.033333 = 0.003333$$

### e) Reconstrucția traseului

In [4]:
path = [int(delta[-1].argmax())]
for t in range(T - 1, 0, -1):
    path.append(int(psi[t][path[-1]]))
path.reverse()

p_joint = delta[-1].max()
print("Note:     ", notes)
print("Secventa: ", [states[s] for s in path])
print("max P(x, o) =", p_joint)

Note:      ['FB', 'FB', 'S', 'B', 'B', 'S', 'B', 'B', 'NS', 'B', 'B', 'S']
Secventa:  ['D', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
max P(x, o) = 2.1093750000000005e-11


### f) Interpretare

Viterbi maximizează probabilitatea comună $P(x_1,\dots,x_{12},o_1,\dots,o_{12})$ peste *toate secvențele complete* de stări. Alegerea celei mai probabile stări separat la fiecare test (maximul marginalei $P(X_t=j\mid o_{1:12})$) poate da o altă secvență, chiar una cu probabilitate comună mai mică sau zero. De exemplu, pentru că $a_{DD}=0$, două teste dificile consecutive sunt imposibile, deși marginalele ar putea indica D de două ori la rând. Viterbi ține cont de tranzițiile dintre teste, nu doar de fiecare test izolat.

Valoarea de la punctul e) este o probabilitate comună $P(x^*,o)$, deci în general foarte mică. Nu este probabilitatea condiționată $P(x^*\mid o)=P(x^*,o)/P(o)$, adică încrederea în traseu dat fiind șirul notelor.

Traseul nu este unic. La primul test, $\delta_1(D)\,a_{DU}=\delta_1(U)\,a_{UU}$ (0.0333·0.5 = 0.0667·0.25), deci secvența care începe cu U (U, U, D, U, D, M, ...) are exact aceeași probabilitate comună. Codul alege D, deoarece `argmax` returnează prima valoare în caz de egalitate.

### Opțional – Forward, $P(x^*\mid o)$ și comparația cu `hmmlearn`

In [5]:
alpha = np.zeros((T, N))
alpha[0] = pi * B[:, o[0]]
for t in range(1, T):
    alpha[t] = B[:, o[t]] * (alpha[t - 1] @ A)

p_obs = alpha[-1].sum()
print("P(o) =", p_obs)
print("P(x* | o) =", p_joint / p_obs)

P(o) = 7.325942897172459e-08
P(x* | o) = 0.0002879322197302604


In [6]:
from hmmlearn import hmm

model = hmm.CategoricalHMM(n_components=3)
model.startprob_ = pi
model.transmat_ = A
model.emissionprob_ = B

X = np.array(o).reshape(-1, 1)
logprob, seq = model.decode(X, algorithm="viterbi")
print("hmmlearn:", [states[s] for s in seq])
print("P(x*, o) =", np.exp(logprob))
print("P(o) =", np.exp(model.score(X)))

hmmlearn: ['U', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
P(x*, o) = 2.109375000000024e-11
P(o) = 7.32594289717248e-08


`hmmlearn` obține aceeași probabilitate comună maximă și același $P(o)$. Secvența lui începe cu U în loc de D, deoarece cele două variante au scor egal (vezi observația de la f), iar biblioteca rezolvă egalitatea altfel.